# VectrixDB with Databricks

VectrixDB integrates with Databricks through two powerful storage backends:
- **Delta Lake**: For governed, ACID-compliant vector storage with Unity Catalog
- **Lakebase**: Databricks managed PostgreSQL with pgvector for high-performance queries

## Delta Lake Storage

Store vectors in Delta Lake tables with Unity Catalog governance.

In [ ]:
from vectrixdb import VectrixDB, StorageConfig, StorageBackend

# Configure Delta Lake storage
delta_config = StorageConfig(
    backend=StorageBackend.DELTA_LAKE,
    delta_workspace_url="https://your-workspace.cloud.databricks.com",
    delta_token="your-databricks-token",
    delta_catalog="main",
    delta_schema="vectrixdb"
)

# Create database with Delta Lake backend
db_delta = VectrixDB(storage_config=delta_config)

print("Connected to Delta Lake!")

## Create Collection in Delta Lake

In [ ]:
# Create a collection - stored as Delta table
collection = db_delta.create_collection(
    name="products",
    dimension=384
)

# Add vectors
collection.add(
    ids=["prod_1", "prod_2", "prod_3"],
    vectors=[
        [0.1] * 384,
        [0.2] * 384,
        [0.3] * 384
    ],
    metadata=[
        {"name": "iPhone", "category": "electronics"},
        {"name": "MacBook", "category": "electronics"},
        {"name": "iPad", "category": "electronics"}
    ]
)

print(f"Added {collection.count()} vectors to Delta Lake")

## Lakebase Storage

Use Databricks Lakebase (managed PostgreSQL with pgvector) for fast vector search.

In [ ]:
# Configure Lakebase storage
lakebase_config = StorageConfig(
    backend=StorageBackend.LAKEBASE,
    lakebase_host="your-lakebase-endpoint.cloud.databricks.com",
    lakebase_port=5432,
    lakebase_database="vectrixdb",
    lakebase_user="your-username",
    lakebase_password="your-password"
)

# Create database with Lakebase backend
db_lakebase = VectrixDB(storage_config=lakebase_config)

print("Connected to Lakebase!")

## High-Performance Search with Lakebase

In [ ]:
# Create collection in Lakebase
products = db_lakebase.create_collection(
    name="products",
    dimension=384
)

# Add vectors
products.add(
    ids=["p1", "p2", "p3"],
    vectors=[
        [0.1] * 384,
        [0.2] * 384,
        [0.3] * 384
    ],
    metadata=[
        {"name": "Product A"},
        {"name": "Product B"},
        {"name": "Product C"}
    ]
)

# Fast vector search
results = products.search(
    query=[0.15] * 384,
    limit=2
)

print("Search Results:")
for r in results.results:
    print(f"  ID: {r.id}, Score: {r.score:.4f}")

## Environment Variables Configuration

For production, configure credentials via environment variables:

In [ ]:
import os

# Delta Lake environment variables
# os.environ["VECTRIXDB_DELTA_WORKSPACE_URL"] = "https://your-workspace.cloud.databricks.com"
# os.environ["VECTRIXDB_DELTA_TOKEN"] = "your-token"
# os.environ["VECTRIXDB_DELTA_CATALOG"] = "main"
# os.environ["VECTRIXDB_DELTA_SCHEMA"] = "vectrixdb"

# Lakebase environment variables
# os.environ["VECTRIXDB_LAKEBASE_HOST"] = "your-host"
# os.environ["VECTRIXDB_LAKEBASE_PORT"] = "5432"
# os.environ["VECTRIXDB_LAKEBASE_DATABASE"] = "vectrixdb"
# os.environ["VECTRIXDB_LAKEBASE_USER"] = "user"
# os.environ["VECTRIXDB_LAKEBASE_PASSWORD"] = "password"

print("Environment variables (uncomment to use)")

## Running in Databricks Notebook

When running directly in a Databricks notebook, VectrixDB can auto-detect credentials:

In [ ]:
# In Databricks notebook - auto-detects workspace context
# from vectrixdb import VectrixDB, StorageConfig, StorageBackend

# delta_config = StorageConfig(
#     backend=StorageBackend.DELTA_LAKE,
#     delta_catalog="main",  # Uses current workspace
#     delta_schema="vectrixdb"
# )

# db = VectrixDB(storage_config=delta_config)

print("Databricks auto-detection example (run in Databricks)")

## Hybrid Architecture: Delta Lake + Lakebase

The recommended architecture for enterprise deployments:
- **Delta Lake**: Source of truth with governance
- **Lakebase**: Fast query layer
- **VectrixSync**: Keeps them in sync

See the **Sync.ipynb** notebook for details on synchronization.

In [ ]:
# Quick preview of sync setup
from vectrixdb import VectrixSync

# sync = VectrixSync(
#     source=db_delta,      # Delta Lake as source
#     target=db_lakebase    # Lakebase as target
# )

# # One-liner to sync and start scheduler
# sync.auto(interval_minutes=5)

print("See Sync.ipynb for full sync examples")

## Unity Catalog Integration

In [ ]:
# Collections are stored as Unity Catalog tables:
# main.vectrixdb.products
# main.vectrixdb._vectrix_collections

# You can query them with SQL:
# SELECT * FROM main.vectrixdb.products LIMIT 10

# Benefits:
# - Row/column level security
# - Data lineage tracking
# - Audit logs
# - Cross-workspace sharing

print("Unity Catalog provides enterprise governance")

## Cleanup

In [ ]:
# Close connections (if using actual connections)
# db_delta.close()
# db_lakebase.close()

print("Demo complete!")